

# The Brute Force Method
### OPIM 5641 - Business Decision Modeling · Module1

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/drdave-teaching/OPIM5641-notebooks/blob/main/3_BruteForce/The Brute Force Method.ipynb)

*Run me top to bottom - **Runtime → Run all**. Data loads from a stable link, so there's nothing to upload.*

# The Brute Force Method
**Dr. Dave Wanik - Dept. Operations & Information Management - University of Connecticut**

------------------------------------------------

This is your first real optimization model, and we're going to crack it the most honest way there is: try every possible combination and keep the best one. That's brute force. It always gets the right answer... and, as you'll feel for yourself in a few minutes, it can be painfully slow. That slowness is actually the whole reason the rest of this course exists - it's what pushes us toward smarter tools like Simplex.

Along the way you'll meet the four building blocks that *every* optimization model is made of, write your first search with nested for-loops, and get a gut feeling for when brute force is your friend and when it completely falls apart.

*Two quotes I keep coming back to:* George Box said *"all models are wrong, but some are useful,"* and H.L. Mencken said *"for every complex problem there is an answer that is clear, simple - and wrong."* A good model lives right between those two - simple enough to be useful, honest enough to be right.

🔷 **The nugget:** brute force is always correct and dead simple - which is exactly why its failure (crushing slowness) is so instructive. It's the 'why' behind every smarter method in this course.

In [ ]:
# In this notebook, I will want to check the amount of time spent in the execution of each cell, so I need to install an additional package
!pip install ipython-autotime
%load_ext autotime

The autotime extension is already loaded. To reload it, use:
  %reload_ext autotime
time: 2.41 s (started: 2025-01-19 15:37:21 +00:00)


## The four building blocks of a model

Here's the big secret of the whole course: every optimization problem, no matter how fancy it looks, is built out of the same four pieces. Once you can spot them in a word problem, you can model almost anything.

1. **Decision variables** - the stuff *you* get to choose (how many chairs to build, where to put a warehouse). These are the `x` and `y` from your old math classes.
2. **The objective** - the one number you're trying to make as big as possible (profit) or as small as possible (cost). You only ever optimize *one* thing.
3. **Constraints** - the rules of reality (you've only got 1,850 fabrication hours; you can't build a negative number of desks).
4. **Data (parameters)** - the numbers you're handed (profit per item, hours per unit). You don't get to change these, you just use them.

**Caution:** parameters usually boil something messy in the real world down to a single number. That's fine - remember Box - just don't mix up a parameter (given, fixed) with a variable (yours to pick).

## Our running example: Veerman Furniture

Here's the problem we'll keep coming back to all semester. The Veerman Furniture Company makes three products - **chairs, desks, and tables** - and each one needs time in three departments: fabrication, assembly, and shipping. The distributor tells us the most we can sell of each (demand), and accounting tells us the profit per item. Our job is to decide how many of each to build to make the most profit this quarter, without running out of hours in any department.

| Department | Chairs | Desks | Tables | Hours available |
|---|---|---|---|---|
| Fabrication | 4 | 6 | 2 | 1,850 |
| Assembly | 3 | 5 | 7 | 2,400 |
| Shipping | 3 | 2 | 4 | 1,500 |
| **Demand potential** | 360 | 300 | 100 | |
| **Profit (USD)** | 15 | 24 | 18 | |

Written out in math, the whole thing is just:

$$\max_{c,d,t}\; 15c + 24d + 18t \quad\text{s.t.}\quad
\begin{cases}
4c + 6d + 2t \le 1850 & \text{(fabrication)}\\
3c + 5d + 7t \le 2400 & \text{(assembly)}\\
3c + 2d + 4t \le 1500 & \text{(shipping)}\\
0 \le c \le 360,\; 0 \le d \le 300,\; 0 \le t \le 100
\end{cases}$$

Don't let the math spook you - that's just the four building blocks written in shorthand.

### Finding the four pieces in Veerman

- **Data:** every number in that table - profits, hours per unit, hours available, demand. Fixed; we just use it.
- **Variables:** how many chairs `c`, desks `d`, and tables `t` to build. One per product.
- **Objective:** maximize profit, $15c + 24d + 18t$.
- **Constraints:** stay under the fabrication, assembly, and shipping hours, don't build more than we can sell, and - the one everybody forgets - you can't build a negative number of anything.

## What brute force actually means

Almost any optimization problem can be solved with one gloriously dumb procedure:

1. For each decision variable, list every value it could possibly take (chairs from 0 to 360, and so on).
2. Make every possible combination of those values.
3. Go through them one at a time. For each one, check whether it's feasible (does it break any constraint?), and if it survives, compute the profit. Keep track of the best you've seen so far.

When the loop finishes, whatever you're holding onto as "the best" *is* the optimal answer. No cleverness required - just a computer and a little patience.

### Brute force on Veerman

Our variables only take whole-number values over a small range, so brute force is easy to set up here:

- Chairs: 0 to 360
- Desks: 0 to 300
- Tables: 0 to 100

We'll make every triple $(c, d, t)$, throw out the ones that break a constraint, and keep the feasible one with the biggest profit. Before we write the whole thing, let's warm up on the Python we'll need.

## A quick Python warm-up

If your Python is rusty, this section is your on-ramp - comments, printing, variables, lists, dictionaries, and loops with a little logic mixed in. If you're already comfortable, skim it and keep moving.

**On your own:**
1. Work out the fabrication hours for 10 chairs, 5 desks, and 2 tables - just hard-code it, and print a sentence with the answer.
2. Now do the same thing using variables for the number of chairs, desks, and tables.

In [1]:
c = 10
title = 'Number of hours:'
print(title, 4*c+5*6+2*2  )




Number of hours: 74


### Lists and dictionaries

We'll stash the problem data in **lists** (ordered, in square brackets) and **dictionaries** (`{key: value}` lookups). Getting in the habit of keeping your *data* separate from your *algorithm* pays off fast - you'll see exactly why in a minute.

In [ ]:
# list
# https://en.wikipedia.org/wiki/Array_data_structure
products = ["chair","desk","table"]

# dictionary
# https://en.wikipedia.org/wiki/Hash_table

fabrication = {
    "chair" : 4,
    "desk" : 6,
    "table" : 2
}
assembly = {
    "chair" : 3,
    "desk" : 5,
    "table" : 7
}
shipping = {
    "chair" : 3,
    "desk" : 2,
    "table" : 4
}
demand = {
    "chair" : 360,
    "desk" : 300,
    "table" : 100
}
profit = {
    "chair" : 15,
    "desk" : 24,
    "table" : 18
}
hours = {
    "fabrication" : 1850,
    "assembly" : 2400,
    "shipping" : 1500
}

time: 811 µs (started: 2025-01-19 15:37:23 +00:00)


**On your own:**
3. Using your variables and the dictionary, work out the fabrication hours for 10 chairs, 5 desks, and 2 tables.

In [ ]:
chairs = 10
desks = 5
table = 2
print("Number of hours:",chairs*fabrication['chair'] + desks*fabrication['desk'] + tables*fabrication['table'])

**On your own:**
4. Try a few different production plans and check whether each one stays under the fabrication-hours limit.

### Logic and filtering

Loops plus `if` statements are the engine of brute force - we sweep across all the candidates and *filter* down to the ones that are actually feasible.

**On your own:** work these in the empty cells below, then run the next cell to check yourself.
5. Print the profit from producing $c$ chairs, for $0 \le c \le 5$.
6. Print the profit from producing $d$ desks, for $0 \le d \le 5$.
7. Print the profit from producing $c$ chairs *and* $d$ desks, for $0 \le c, d \le 5$.

**On your own:**
8. Build and print a list of the profits for producing $c$ chairs and $d$ desks, for every combination where $c + d \le 5$.

## The full brute-force search

Now we put it all together: load the data, loop over every $(c, d, t)$, keep the feasible ones, and hang onto the winner.

### The data

Here's our data in lists and dictionaries. Notice how the data sits completely apart from the search logic - that separation is about to earn its keep.

In [ ]:
# list
# https://en.wikipedia.org/wiki/Array_data_structure
products = ["chair","desk","table"]

# dictionary
# https://en.wikipedia.org/wiki/Hash_table

fabrication = {
    "chair" : 4,
    "desk" : 6,
    "table" : 2
}
assembly = {
    "chair" : 3,
    "desk" : 5,
    "table" : 7
}
shipping = {
    "chair" : 3,
    "desk" : 2,
    "table" : 4
}
demand = {
    "chair" : 360,
    "desk" : 300,
    "table" : 100
}
profit = {
    "chair" : 15,
    "desk" : 24,
    "table" : 18
}
hours = {
    "fabrication" : 1850,
    "assembly" : 2400,
    "shipping" : 1500
}

time: 4.23 ms (started: 2023-02-03 13:38:22 +00:00)


In [ ]:
# INITIALIZATION: Variables containing current best value and current best solution
best_profit = 0
best_n_chairs = 0
best_n_desks = 0
best_n_tables = 0
# FOR LOOPS: one for each variable
for chairs in range(0,demand["chair"]+1):
  for desks in range(0,demand["desk"]+1):
    for tables in range(0,demand["table"]+1):

       # CHECK CONSTRAINTS
       # Check required fabrication hours
        if fabrication["chair"]*chairs + fabrication["desk"]*desks + fabrication["table"]*tables > hours["fabrication"]:
          continue
        # Check required assembly hours
        if assembly["chair"]*chairs + assembly["desk"]*desks + assembly["table"]*tables > hours["assembly"]:
          continue
        # Check required shipping hours
        if shipping["chair"]*chairs + shipping["desk"]*desks + shipping["table"]*tables > hours["shipping"]:
          continue

        # UPDATE OBJECTIVE
        if profit["chair"]*chairs + profit["desk"]*desks + profit["table"]*tables > best_profit:
          best_profit = profit["chair"]*chairs + profit["desk"]*desks + profit["table"]*tables
          best_n_chairs = chairs
          best_n_desks = desks
          best_n_tables = tables
          print(best_profit, best_n_chairs, best_n_desks, best_n_tables)
# PRINT RESULTS
print("Maximum profit:",best_profit)
print("Chairs",best_n_chairs)
print("Desks",best_n_desks)
print("Tables",best_n_tables)

18 0 0 1
36 0 0 2
54 0 0 3
72 0 0 4
90 0 0 5
108 0 0 6
126 0 0 7
144 0 0 8
162 0 0 9
180 0 0 10
198 0 0 11
216 0 0 12
234 0 0 13
252 0 0 14
270 0 0 15
288 0 0 16
306 0 0 17
324 0 0 18
342 0 0 19
360 0 0 20
378 0 0 21
396 0 0 22
414 0 0 23
432 0 0 24
450 0 0 25
468 0 0 26
486 0 0 27
504 0 0 28
522 0 0 29
540 0 0 30
558 0 0 31
576 0 0 32
594 0 0 33
612 0 0 34
630 0 0 35
648 0 0 36
666 0 0 37
684 0 0 38
702 0 0 39
720 0 0 40
738 0 0 41
756 0 0 42
774 0 0 43
792 0 0 44
810 0 0 45
828 0 0 46
846 0 0 47
864 0 0 48
882 0 0 49
900 0 0 50
918 0 0 51
936 0 0 52
954 0 0 53
972 0 0 54
990 0 0 55
1008 0 0 56
1026 0 0 57
1044 0 0 58
1062 0 0 59
1080 0 0 60
1098 0 0 61
1116 0 0 62
1134 0 0 63
1152 0 0 64
1170 0 0 65
1188 0 0 66
1206 0 0 67
1224 0 0 68
1242 0 0 69
1260 0 0 70
1278 0 0 71
1296 0 0 72
1314 0 0 73
1332 0 0 74
1350 0 0 75
1368 0 0 76
1386 0 0 77
1404 0 0 78
1422 0 0 79
1440 0 0 80
1458 0 0 81
1476 0 0 82
1494 0 0 83
1512 0 0 84
1530 0 0 85
1548 0 0 86
1566 0 0 87
1584 0 0 88
1602 0 0 89
1

## Why brute force is great (sometimes)

We just solved a real product-mix problem with a plain old loop, and honestly that's good news - a *lot* of problems give in to exactly this approach. What's to love:

- **It's dead simple.** You can explain it to a five-year-old or to a CEO and both will nod along. Try everything, keep the best.
- **It's easy to write** any time you can list out (or reasonably approximate) the values each variable can take.
- **It needs no clever ideas** - no fancy math, no special algorithm.

## Why brute force falls apart

Now watch this. We'll solve the *exact same problem*, but double all the demands and the available hours:

| Department | Chairs | Desks | Tables | Hours available |
|---|---|---|---|---|
| Fabrication | 4 | 6 | 2 | 3,700 |
| Assembly | 3 | 5 | 7 | 4,800 |
| Shipping | 3 | 2 | 4 | 3,000 |
| **Demand potential** | 720 | 600 | 200 | |
| **Profit (USD)** | 15 | 24 | 18 | |

Run it and keep an eye on how long it takes.

In [ ]:
# list
# https://en.wikipedia.org/wiki/Array_data_structure
products = ["chair","desk","table"]

# dictionary
# https://en.wikipedia.org/wiki/Hash_table

fabrication = {
    "chair" : 4,
    "desk" : 6,
    "table" : 2
}
assembly = {
    "chair" : 3,
    "desk" : 5,
    "table" : 7
}
shipping = {
    "chair" : 3,
    "desk" : 2,
    "table" : 4
}
# You need to change demands (multiply by 2)
demand = {
    "chair" : 720,
    "desk" : 600,
    "table" : 200
}
profit = {
    "chair" : 15,
    "desk" : 24,
    "table" : 18
}
# You need to change hours (multiply by 2)
hours = {
    "fabrication" : 3700,
    "assembly" : 4800,
    "shipping" : 3000
}

time: 1.32 ms (started: 2023-02-03 13:38:44 +00:00)


Notice the code below is **exactly the same** as before - only the data changed. That's the payoff for keeping data and algorithm separate: to solve a brand new version, you just swap the numbers and hit run.

In [ ]:
# Variables containing current best value and current best solution
best_profit = 0
best_n_chairs = 0
best_n_desks = 0
best_n_tables = 0
for chairs in range(0,demand["chair"]+1):
  for desks in range(0,demand["desk"]+1):
    for tables in range(0,demand["table"]+1):
       # Check required fabrication hours
        if fabrication["chair"]*chairs + fabrication["desk"]*desks + fabrication["table"]*tables > hours["fabrication"]:
          continue
        # Check required assembly hours
        if assembly["chair"]*chairs + assembly["desk"]*desks + assembly["table"]*tables > hours["assembly"]:
          continue
        # Check required shipping hours
        if shipping["chair"]*chairs + shipping["desk"]*desks + shipping["table"]*tables > hours["shipping"]:
          continue
        if profit["chair"]*chairs + profit["desk"]*desks + profit["table"]*tables > best_profit:
          best_profit = profit["chair"]*chairs + profit["desk"]*desks + profit["table"]*tables
          best_n_chairs = chairs
          best_n_desks = desks
          best_n_tables = tables

print("Maximum profit:",best_profit)
print("Chairs",best_n_chairs)
print("Desks",best_n_desks)
print("Tables",best_n_tables)

Maximum profit: 16800
Chairs 0
Desks 550
Tables 200
time: 1min 28s (started: 2023-02-03 13:39:49 +00:00)


## The running time explodes

The two problems are basically identical - the best plan just doubled - and yet brute force took about **8 times longer** on the bigger one. Bump the inputs by 10x instead of 2x and you'd be looking at roughly **1,000 times longer**, which is hours of your life you're not getting back.

**Caution:** you can sometimes make brute force a little smarter - say, stop checking tables once chairs and desks already blow the fabrication limit. That helps at the edges, but it doesn't save you. Some problems are just too big to enumerate, full stop.

## A few other things worth knowing

- **Continuous variables break it.** If a variable can be *any* real number instead of a whole number, you can't list out its values - there are infinitely many of them.
- **You can quit early.** Sometimes you just need a *good enough* (or even just *feasible*) answer, so you stop the moment you're happy. Handy... but -
- **finding even one feasible answer can take forever.** If the problem has no feasible solution at all, brute force has to check the whole space just to prove it.

## So when should you actually use it?

- As a **sanity check** - not sure your model is right? Run a quick brute-force pass on a small version and see if the answer makes sense.
- On **small** problems it can work just fine (like the original Veerman problem did).
- On **real** problems it usually isn't good enough - which is exactly why Simplex and the other methods in this course exist.

## On your own: the Machine Assignment problem

Say we've got **4 machines** and **4 jobs**, and the time to finish each job depends on which machine does it:

| Time | J1 | J2 | J3 | J4 |
|---|---|---|---|---|
| M1 | 20 | 15 | 11 | 16 |
| M2 | 15 | 13 | 5 | 10 |
| M3 | 22 | 14 | 9 | 9 |
| M4 | 18 | 14 | 7 | 11 |

**Which machine should do which job so we finish everything in the least total time?** The rules: each machine does exactly one job, and each job gets done by exactly one machine. (It's the same idea as assigning employees to tasks.)

We'll list out the possibilities with Python's `itertools` - every **permutation** is one way to line the machines up against the jobs.

In [ ]:
# Generate all possible permutations of assignments
from itertools import permutations

time_table = [
              [20, 15, 11, 16],
              [15, 13, 5, 10],
              [22, 14, 9, 9],
              [18, 14, 7, 11]
]
machines = [0,1,2,3]

# This list of lists will contain each possible arrangement of jobs in machines
permutations = list(permutations(machines))
print(permutations)

[(0, 1, 2, 3), (0, 1, 3, 2), (0, 2, 1, 3), (0, 2, 3, 1), (0, 3, 1, 2), (0, 3, 2, 1), (1, 0, 2, 3), (1, 0, 3, 2), (1, 2, 0, 3), (1, 2, 3, 0), (1, 3, 0, 2), (1, 3, 2, 0), (2, 0, 1, 3), (2, 0, 3, 1), (2, 1, 0, 3), (2, 1, 3, 0), (2, 3, 0, 1), (2, 3, 1, 0), (3, 0, 1, 2), (3, 0, 2, 1), (3, 1, 0, 2), (3, 1, 2, 0), (3, 2, 0, 1), (3, 2, 1, 0)]
time: 1.92 ms (started: 2023-02-03 13:43:02 +00:00)


In [ ]:
best_time = 1000
best_permutation = 0
for permutation in permutations:
  total_time = 0
  for i in permutation:
    total_time += time_table[permutation[i]][i]
  if total_time < best_time:
    best_time = total_time
    best_permutation = permutation

print("Minimum total time:",best_time)
print("Best combination:",best_permutation)



Minimum total time: 46
Best combination: (1, 0, 3, 2)
time: 4.32 ms (started: 2023-02-03 13:43:15 +00:00)


Brute force crushes this one because it's tiny - even with bigger times it stays fast. But start adding machines and jobs and it detonates. With 70 machines and 70 jobs, the number of permutations is about a **googol** ($10^{100}$). For perspective, there are only about $10^{80}$ atoms in the observable universe - so you'd run out of universe long before you finished checking every option. That right there is the gut-punch that makes us go looking for a better way.

## If you want to go deeper

This course is mostly about **exact** methods - the ones that hand you the provably best answer. Brute force counts as exact when all your variables are whole numbers. Out in the wild, people also reach for smarter ideas, and they come in three flavors:

- **Exact combinatorial algorithms** - give you the exact answer fast, without checking everything. Rare, but wonderful when they exist.
- **Approximation algorithms** - trade a little accuracy for a lot of speed, with a guarantee on how far off they can be.
- **Heuristics** - clever rules of thumb (think genetic algorithms or simulated annealing) with no guarantee, but often great in practice.

We'll spend most of our time on **linear programming and Simplex** - exact, fast, and the real workhorse of decision modeling.

## Wrapping up

- Every optimization model is just four pieces: **variables, an objective, constraints, and data.**
- **Brute force** means listing every candidate and keeping the best feasible one. Always right, always simple.
- It's also **brutally slow** - the running time blows up as the problem grows, and it can't handle continuous variables at all.
- That slowness is the whole reason for the rest of the course. Next up: the **graphical method**, where we actually *see* the answer for two-variable problems, and then **Simplex**, which scales it up.